In [2]:
from pathlib import Path
import re
import pandas as pd
from datetime import datetime

In [3]:
PROJECT_ROOT = Path.cwd().parents[1]

INPUT_MARKDOWN_DIR = PROJECT_ROOT / "markdown" / "raw"
OUTPUT_MARKDOWN_DIR = PROJECT_ROOT / "markdown" / "cleaned"
LOG_DIR = PROJECT_ROOT / "progress" / "markdown_cleaning" / "logs"

OUTPUT_MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input markdown folder:", INPUT_MARKDOWN_DIR)
print("Output cleaned markdown folder:", OUTPUT_MARKDOWN_DIR)
print("Log folder:", LOG_DIR)

Project root: /home/nsirim/Github/mscdsa/msc
Input markdown folder: /home/nsirim/Github/mscdsa/msc/markdown/raw
Output cleaned markdown folder: /home/nsirim/Github/mscdsa/msc/markdown/cleaned
Log folder: /home/nsirim/Github/mscdsa/msc/progress/markdown_cleaning/logs


In [4]:
def should_skip_path(path: Path) -> bool:
    parts = set(path.parts)

    if "cleaned" in parts:
        return True

    if path.suffix.lower() != ".md":
        return True

    if any(part.endswith("_artifacts") for part in path.parts):
        return True

    return False


markdown_files = [
    path for path in INPUT_MARKDOWN_DIR.rglob("*.md")
    if not should_skip_path(path)
]

markdown_files = sorted(markdown_files)

print(f"Found {len(markdown_files)} Markdown files.")

for path in markdown_files:
    print(path.relative_to(PROJECT_ROOT))

Found 73 Markdown files.
markdown/raw/policy/AI competency framework  for students.md
markdown/raw/policy/AI competency framework  for teachers.md
markdown/raw/policy/australia/AHISA MEMBER SURVEY REPORT Gen AI in Australian independent schools.md
markdown/raw/policy/australia/AI systems in teaching and learning_ Principles and practical exa.md
markdown/raw/policy/australia/ASC-Artificial-Intelligence-Policy-2025.md
markdown/raw/policy/australia/Artificial Intelligence Policy.md
markdown/raw/policy/australia/Australian Framework for Generative AI in Schools.md
markdown/raw/policy/australia/DOAI-24-Educator-Guide.md
markdown/raw/policy/australia/Generative_AI_Usage_Policy.md
markdown/raw/policy/australia/Generative_AI_in_the_Australian_education_system_An_open_data_set_of_stakeholder_recommendations_and_emerging_analysis _from_a_public_inquiry.md
markdown/raw/policy/australia/POLICY TITLE: Artificial Intelligence (AI) Usage Policy for Staff, Teachers and Students.md
markdown/raw/policy/

In [5]:
sample_file = markdown_files[0]

print("Sample file:")
print(sample_file.relative_to(PROJECT_ROOT))

raw_text = sample_file.read_text(encoding="utf-8", errors="replace")

print("\nFirst 3000 characters:\n")
print(raw_text[:3000])

Sample file:
markdown/raw/policy/AI competency framework  for students.md

First 3000 characters:



## AI competency framework

## for students





## UNESCO - a global leader in education

Education is UNESCO's top priority because it is a basic human right and the foundation for peace and sustainable development. UNESCO is the United Nations' specialized agency for education, providing global and regional leadership to drive progress, strengthening the resilience and capacity of national systems to serve all learners. UNESCO also leads efforts to respond to contemporary global challenges through transformative learning, with special focus on gender equality and Africa across all actions.



## The Global Education 2030 Agenda

UNESCO, as the United Nations' specialized agency for education, is entrusted to lead and coordinate the Education 2030 Agenda, which is part of a global movement to eradicate poverty through 17 Sustainable Development Goals by 2030. Education, essential to a

In [7]:
def remove_html_comments(text: str) -> str:
    """
    Remove HTML comments such as:
    <!-- Image file: ... -->
    <!-- Long image ID: ... -->
    """
    return re.sub(r"<!--.*?-->", "\n", text, flags=re.DOTALL)

def remove_markdown_image_links(text: str) -> str:
    """
    Remove Markdown image links, including escaped image links.

    Examples removed:
    ![Image](path/to/image.png)
    \\![Missing image: title](path/to/image.png)
    """
    return re.sub(
        r"^\s*\\?!\[[^\]]*\]\([^)]+\)\s*$",
        "",
        text,
        flags=re.MULTILINE
    )

def remove_raw_image_paths(text: str) -> str:
    """
    Remove standalone local image paths if Docling inserted them as text.
    """
    text = re.sub(
        r"/home/[^\s)]+?\.(png|jpg|jpeg|webp)",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"[A-Za-z]:\\[^\s)]+?\.(png|jpg|jpeg|webp)",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text

def remove_page_number_lines(text: str) -> str:
    """
    Remove lines that contain only page numbers.
    """
    return re.sub(r"(?m)^\s*\d{1,4}\s*$", "", text)


def remove_dot_leader_lines(text: str) -> str:
    """
    Remove table-of-contents style lines.

    Example:
    Overview................................................4
    """
    cleaned_lines = []

    for line in text.splitlines():
        stripped = line.strip()

        if re.search(r"\.{8,}\s*\d+\s*$", stripped):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

def is_probably_toc_table(table_lines: list[str]) -> bool:
    """
    Detect Markdown tables that are probably tables of contents.
    We remove tables with many dotted leaders.
    """
    table_text = "\n".join(table_lines)

    dot_leader_count = len(re.findall(r"\.{8,}", table_text))
    page_ref_count = len(re.findall(r"\.{4,}\s*\d+", table_text))

    if dot_leader_count >= 2 or page_ref_count >= 2:
        return True

    rows = [line for line in table_lines if "|" in line]

    if not rows:
        return False

    toc_like_rows = 0

    for row in rows:
        if re.search(r"\.{4,}\s*\d+", row):
            toc_like_rows += 1

    if len(rows) > 0 and toc_like_rows / len(rows) > 0.30:
        return True

    return False

def remove_toc_tables(text: str) -> str:
    """
    Remove only table-of-contents-like Markdown tables.
    Keep normal tables.
    """
    lines = text.splitlines()
    output = []
    i = 0

    while i < len(lines):
        line = lines[i]

        if "|" in line:
            table_block = []
            j = i

            while j < len(lines) and "|" in lines[j]:
                table_block.append(lines[j])
                j += 1

            if len(table_block) >= 2 and is_probably_toc_table(table_block):
                output.append("")
                i = j
                continue

            output.extend(table_block)
            i = j
            continue

        output.append(line)
        i += 1

    return "\n".join(output)

def clean_markdown_syntax_noise(text: str) -> str:
    """
    Remove Markdown syntax that adds noise for NLP,
    while preserving headings, text and normal tables.
    """
    # Remove code block fences but keep inner text.
    text = re.sub(r"```[\w-]*", "", text)

    # Remove emphasis markers but keep words.
    text = text.replace("**", "")
    text = text.replace("__", "")

    # Remove decorative horizontal rules.
    text = re.sub(r"(?m)^\s*[-*_]{3,}\s*$", "", text)

    return text


def remove_duplicate_consecutive_headings(text: str) -> str:
    """
    Remove duplicate consecutive headings.

    Example:
    ## Title
    ## Title
    """
    lines = text.splitlines()
    output = []
    previous_heading = None

    heading_pattern = re.compile(r"^(#{1,6})\s+(.*)$")

    for line in lines:
        match = heading_pattern.match(line.strip())

        if match:
            heading_text = match.group(2).strip().lower()

            if previous_heading == heading_text:
                continue

            previous_heading = heading_text
            output.append(line)
        else:
            if line.strip():
                previous_heading = None
            output.append(line)

    return "\n".join(output)


def remove_repeated_document_title(text: str) -> str:
    """
    Remove repeated first title if it appears immediately again.
    """
    lines = text.splitlines()
    heading_pattern = re.compile(r"^(#{1,6})\s+(.*)$")

    headings_seen = []

    for i, line in enumerate(lines[:30]):
        match = heading_pattern.match(line.strip())
        if match:
            headings_seen.append((i, match.group(2).strip().lower()))

    if len(headings_seen) >= 2:
        first_idx, first_title = headings_seen[0]
        second_idx, second_title = headings_seen[1]

        if first_title == second_title and second_idx - first_idx <= 6:
            lines.pop(second_idx)

    return "\n".join(lines)

def normalize_first_heading(text: str) -> str:
    """
    Convert the first Markdown heading to #.
    Leave the rest unchanged.
    """
    lines = text.splitlines()
    changed = False

    for idx, line in enumerate(lines):
        match = re.match(r"^(#{1,6})\s+(.*)$", line.strip())

        if match and not changed:
            title = match.group(2).strip()
            lines[idx] = f"# {title}"
            changed = True
            break

    return "\n".join(lines)

def normalize_whitespace(text: str) -> str:
    """
    Normalize spacing and blank lines.
    """
    text = text.replace("\xa0", " ")

    # Remove trailing whitespace.
    text = "\n".join(line.rstrip() for line in text.splitlines())

    normalized_lines = []

    for line in text.splitlines():
        if "|" in line:
            # Keep tables readable.
            normalized_lines.append(line.strip())
        else:
            # Collapse internal spaces.
            normalized_lines.append(re.sub(r"[ \t]+", " ", line).strip())

    text = "\n".join(normalized_lines)

    # Collapse many blank lines.
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip() + "\n"

In [6]:
def clean_markdown(text: str) -> str:
    """
    Full cleaning pipeline for Markdown files.
    """
    text = remove_html_comments(text)
    text = remove_markdown_image_links(text)
    text = remove_raw_image_paths(text)

    text = remove_toc_tables(text)
    text = remove_dot_leader_lines(text)
    text = remove_page_number_lines(text)

    text = clean_markdown_syntax_noise(text)
    text = remove_duplicate_consecutive_headings(text)
    text = remove_repeated_document_title(text)
    text = normalize_first_heading(text)

    text = normalize_whitespace(text)

    return text

In [8]:
cleaned_sample = clean_markdown(raw_text)

print("Original length:", len(raw_text))
print("Cleaned length:", len(cleaned_sample))
print("Removed characters:", len(raw_text) - len(cleaned_sample))

print("\nFirst 3000 characters of cleaned version:\n")
print(cleaned_sample[:3000])

Original length: 395968
Cleaned length: 395846
Removed characters: 122

First 3000 characters of cleaned version:

# AI competency framework

## for students

## UNESCO - a global leader in education

Education is UNESCO's top priority because it is a basic human right and the foundation for peace and sustainable development. UNESCO is the United Nations' specialized agency for education, providing global and regional leadership to drive progress, strengthening the resilience and capacity of national systems to serve all learners. UNESCO also leads efforts to respond to contemporary global challenges through transformative learning, with special focus on gender equality and Africa across all actions.

## The Global Education 2030 Agenda

UNESCO, as the United Nations' specialized agency for education, is entrusted to lead and coordinate the Education 2030 Agenda, which is part of a global movement to eradicate poverty through 17 Sustainable Development Goals by 2030. Education, essenti

In [9]:
print("========== RAW SAMPLE ==========\n")
print(raw_text[:1500])

print("\n\n========== CLEANED SAMPLE ==========\n")
print(cleaned_sample[:1500])

========== RAW SAMPLE ==========



## AI competency framework

## for students





## UNESCO - a global leader in education

Education is UNESCO's top priority because it is a basic human right and the foundation for peace and sustainable development. UNESCO is the United Nations' specialized agency for education, providing global and regional leadership to drive progress, strengthening the resilience and capacity of national systems to serve all learners. UNESCO also leads efforts to respond to contemporary global challenges through transformative learning, with special focus on gender equality and Africa across all actions.



## The Global Education 2030 Agenda

UNESCO, as the United Nations' specialized agency for education, is entrusted to lead and coordinate the Education 2030 Agenda, which is part of a global movement to eradicate poverty through 17 Sustainable Development Goals by 2030. Education, essential to achieve all of these goals, has its own dedicated Goal 4, which ai

In [10]:
def output_path_for(input_path: Path) -> Path:
    """
    Mirror the markdown folder structure inside markdown/cleaned.

    Example:
    markdown/policy/france/file.md
    -> markdown/cleaned/policy/france/file.md
    """
    relative = input_path.relative_to(INPUT_MARKDOWN_DIR)
    return OUTPUT_MARKDOWN_DIR / relative

In [11]:
cleaning_report = []

for idx, input_path in enumerate(markdown_files, start=1):
    output_path = output_path_for(input_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    raw_text = input_path.read_text(encoding="utf-8", errors="replace")
    cleaned_text = clean_markdown(raw_text)

    output_path.write_text(cleaned_text, encoding="utf-8")

    cleaning_report.append({
        "input_file": str(input_path.relative_to(PROJECT_ROOT)),
        "output_file": str(output_path.relative_to(PROJECT_ROOT)),
        "raw_chars": len(raw_text),
        "cleaned_chars": len(cleaned_text),
        "removed_chars": len(raw_text) - len(cleaned_text),
        "raw_lines": raw_text.count("\n") + 1,
        "cleaned_lines": cleaned_text.count("\n") + 1,
    })

    if idx % 10 == 0 or idx == len(markdown_files):
        print(f"Processed {idx}/{len(markdown_files)}")

df_report = pd.DataFrame(cleaning_report)

print("Done.")
df_report.head()

Processed 10/73
Processed 20/73
Processed 30/73
Processed 40/73
Processed 50/73
Processed 60/73
Processed 70/73
Processed 73/73
Done.


,input_file,output_file,raw_chars,cleaned_chars,removed_chars,raw_lines,cleaned_lines
0,markdown/raw/policy/AI competency framework f...,markdown/cleaned/policy/AI competency framewor...,395968,395846,122,880,853
1,markdown/raw/policy/AI competency framework f...,markdown/cleaned/policy/AI competency framewor...,321363,321146,217,890,873
2,markdown/raw/policy/australia/AHISA MEMBER SUR...,markdown/cleaned/policy/australia/AHISA MEMBER...,67965,67910,55,641,608
3,markdown/raw/policy/australia/AI systems in te...,markdown/cleaned/policy/australia/AI systems i...,112429,98256,14173,627,571
4,markdown/raw/policy/australia/ASC-Artificial-I...,markdown/cleaned/policy/australia/ASC-Artifici...,10608,10606,2,156,156


In [12]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

report_csv = LOG_DIR / f"clean_markdown_report_{timestamp}.csv"
df_report.to_csv(report_csv, index=False, encoding="utf-8")

print("Saved report to:")
print(report_csv)

Saved report to:
/home/nsirim/Github/mscdsa/msc/progress/markdown_cleaning/logs/clean_markdown_report_20260601_235007.csv


In [13]:
df_report.describe()

,raw_chars,cleaned_chars,removed_chars,raw_lines,cleaned_lines
count,73.000000,73.000000,73.000000,73.000000,73.000000
mean,66288.835616,63213.260274,3075.575342,516.109589,475.739726
std,82171.582255,79022.843752,8106.337356,539.349829,491.318409
min,3545.000000,3544.000000,0.000000,21.000000,16.000000
25%,12535.000000,12164.000000,15.000000,156.000000,156.000000
50%,31827.000000,30890.000000,174.000000,346.000000,316.000000
75%,71822.000000,71679.000000,1240.000000,641.000000,615.000000
max,395968.000000,395846.000000,50271.000000,2355.000000,1928.000000


In [14]:
print("Files cleaned:", len(df_report))
print("Total raw characters:", df_report["raw_chars"].sum())
print("Total cleaned characters:", df_report["cleaned_chars"].sum())
print("Total removed characters:", df_report["removed_chars"].sum())

Files cleaned: 73
Total raw characters: 4839085
Total cleaned characters: 4614568
Total removed characters: 224517


In [15]:
df_report.sort_values("removed_chars", ascending=False).head(50)

,input_file,output_file,raw_chars,cleaned_chars,removed_chars,raw_lines,cleaned_lines
71,markdown/raw/sentiment/generative artificial i...,markdown/cleaned/sentiment/generative artifici...,194621,144350,50271,886,816
21,markdown/raw/policy/france/L’intelligence arti...,markdown/cleaned/policy/france/L’intelligence ...,280637,249671,30966,1787,1719
54,markdown/raw/policy/usa/ai_report_us_gov.md,markdown/cleaned/policy/usa/ai_report_us_gov.md,230661,204979,25682,1398,1303
60,markdown/raw/sentiment/AI4T_WP3_D3.3_NR_Irelan...,markdown/cleaned/sentiment/AI4T_WP3_D3.3_NR_Ir...,241758,221644,20114,2355,1928
3,markdown/raw/policy/australia/AI systems in te...,markdown/cleaned/policy/australia/AI systems i...,112429,98256,14173,627,571
72,markdown/raw/sentiment/qqi_generative-artifici...,markdown/cleaned/sentiment/qqi_generative-arti...,53563,40738,12825,423,377
17,markdown/raw/policy/australia/generative-artif...,markdown/cleaned/policy/australia/generative-a...,35215,22900,12315,341,316
61,markdown/raw/sentiment/AI_views of youth worke...,markdown/cleaned/sentiment/AI_views of youth w...,70881,59210,11671,377,343
49,markdown/raw/policy/usa/K-12 Artificial Intell...,markdown/cleaned/policy/usa/K-12 Artificial In...,44860,36040,8820,429,377
16,markdown/raw/policy/australia/brisbanesde-ai-u...,markdown/cleaned/policy/australia/brisbanesde-...,20324,12620,7704,228,197


In [16]:
cleaned_files = sorted(OUTPUT_MARKDOWN_DIR.rglob("*.md"))

print(f"Cleaned Markdown files: {len(cleaned_files)}")

for path in cleaned_files[:30]:
    print(path.relative_to(PROJECT_ROOT))

Cleaned Markdown files: 73
markdown/cleaned/policy/AI competency framework  for students.md
markdown/cleaned/policy/AI competency framework  for teachers.md
markdown/cleaned/policy/australia/AHISA MEMBER SURVEY REPORT Gen AI in Australian independent schools.md
markdown/cleaned/policy/australia/AI systems in teaching and learning_ Principles and practical exa.md
markdown/cleaned/policy/australia/ASC-Artificial-Intelligence-Policy-2025.md
markdown/cleaned/policy/australia/Artificial Intelligence Policy.md
markdown/cleaned/policy/australia/Australian Framework for Generative AI in Schools.md
markdown/cleaned/policy/australia/DOAI-24-Educator-Guide.md
markdown/cleaned/policy/australia/Generative_AI_Usage_Policy.md
markdown/cleaned/policy/australia/Generative_AI_in_the_Australian_education_system_An_open_data_set_of_stakeholder_recommendations_and_emerging_analysis _from_a_public_inquiry.md
markdown/cleaned/policy/australia/POLICY TITLE: Artificial Intelligence (AI) Usage Policy for Staff,

In [17]:
sample_cleaned_file = cleaned_files[0]

print("Previewing:")
print(sample_cleaned_file.relative_to(PROJECT_ROOT))

text = sample_cleaned_file.read_text(encoding="utf-8", errors="replace")

print(text[:4000])

Previewing:
markdown/cleaned/policy/AI competency framework  for students.md
# AI competency framework

## for students

## UNESCO - a global leader in education

Education is UNESCO's top priority because it is a basic human right and the foundation for peace and sustainable development. UNESCO is the United Nations' specialized agency for education, providing global and regional leadership to drive progress, strengthening the resilience and capacity of national systems to serve all learners. UNESCO also leads efforts to respond to contemporary global challenges through transformative learning, with special focus on gender equality and Africa across all actions.

## The Global Education 2030 Agenda

UNESCO, as the United Nations' specialized agency for education, is entrusted to lead and coordinate the Education 2030 Agenda, which is part of a global movement to eradicate poverty through 17 Sustainable Development Goals by 2030. Education, essential to achieve all of these goals, has 

In [18]:
sample_cleaned_file = cleaned_files[0]

print("Previewing:")
print(sample_cleaned_file.relative_to(PROJECT_ROOT))

text = sample_cleaned_file.read_text(encoding="utf-8", errors="replace")

print(text[:4000])

Previewing:
markdown/cleaned/policy/AI competency framework  for students.md
# AI competency framework

## for students

## UNESCO - a global leader in education

Education is UNESCO's top priority because it is a basic human right and the foundation for peace and sustainable development. UNESCO is the United Nations' specialized agency for education, providing global and regional leadership to drive progress, strengthening the resilience and capacity of national systems to serve all learners. UNESCO also leads efforts to respond to contemporary global challenges through transformative learning, with special focus on gender equality and Africa across all actions.

## The Global Education 2030 Agenda

UNESCO, as the United Nations' specialized agency for education, is entrusted to lead and coordinate the Education 2030 Agenda, which is part of a global movement to eradicate poverty through 17 Sustainable Development Goals by 2030. Education, essential to achieve all of these goals, has 

In [19]:
remaining_local_paths = []

for path in cleaned_files:
    text = path.read_text(encoding="utf-8", errors="replace")

    if re.search(r"/home/[^\s)]+?\.(png|jpg|jpeg|webp)", text, flags=re.IGNORECASE):
        remaining_local_paths.append(path)

print("Files with remaining local image paths:", len(remaining_local_paths))

for path in remaining_local_paths[:20]:
    print(path.relative_to(PROJECT_ROOT))

Files with remaining local image paths: 0


In [20]:
short_files = []

for path in cleaned_files:
    text = path.read_text(encoding="utf-8", errors="replace")
    words = len(re.findall(r"\b\w+\b", text))

    short_files.append({
        "file": str(path.relative_to(PROJECT_ROOT)),
        "words": words,
        "chars": len(text)
    })

df_short = pd.DataFrame(short_files).sort_values("words")

df_short.head(20)

,file,words,chars
11,markdown/cleaned/policy/australia/Review of th...,505,3544
13,markdown/cleaned/policy/australia/Use-of-Gener...,536,3686
26,markdown/cleaned/policy/ireland/2025-11-18_ope...,694,4167
18,markdown/cleaned/policy/australia/generative-a...,718,5856
29,markdown/cleaned/policy/ireland/2026-01-20_ope...,776,4833
30,markdown/cleaned/policy/ireland/2026-01-20_ope...,820,5211
20,markdown/cleaned/policy/france/AI Policy and ...,890,5589
65,markdown/cleaned/sentiment/How do the French r...,1015,6312
27,markdown/cleaned/policy/ireland/2025-11-18_ope...,1041,6813
31,markdown/cleaned/policy/ireland/2026-01-20_ope...,1146,7269
